# Model Feature Generation
Loads the base processed reviews, generates model-driven features (sentiment, emotion, topics), and saves a single enriched file for downstream analysis.

In [41]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

In [42]:
# --------------------------------------------
# Config
# --------------------------------------------
INPUT_FILE = os.getenv("INPUT_FILE", "Processed_Reviews.csv")  # single source; load and save to same file
BATCH_SENTIMENT = 32
BATCH_EMOTION = 32
MAX_TEXT_LEN = 256
TOPIC_SAMPLE_N = None  # set to an int (e.g., 3000) to test topics on a subset first
device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

Using device: CPU


In [43]:
# --------------------------------------------
# 1. Load base dataset
# --------------------------------------------
df = pd.read_csv(INPUT_FILE)
print(f"Loaded: {INPUT_FILE}")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = [
    "Title", "Text", "Rating_Class",
    "Rating", "Location_Name", "Located_City", "Location_Type",
    "Province", "District", "User_Country", "User_Region",
    "Travel_Date_Month", "Travel_Date_Year",
    "Published_Date_Month", "Published_Date_Year",
    "User_Contributions", "Helpful_Votes",
    "Review_Length", "Title_Length", "Review_Delay_Days"
]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in base file: {missing_cols}")

Loaded: Processed_Reviews.csv
Shape: (16156, 22)
Columns: ['Unnamed: 0', 'Location_Name', 'Located_City', 'Province', 'District', 'Location_Type', 'User_Locale', 'User_Country', 'User_Region', 'Travel_Date_Month', 'Travel_Date_Year', 'Published_Date_Month', 'Published_Date_Year', 'User_Contributions', 'Rating', 'Helpful_Votes', 'Title', 'Text', 'Review_Length', 'Title_Length', 'Rating_Class', 'Review_Delay_Days']


In [44]:
# --------------------------------------------
# 2. Create Combined_Text
# --------------------------------------------
df["Title"] = df["Title"].fillna("").astype(str)
df["Text"] = df["Text"].fillna("").astype(str)
df["Combined_Text"] = (df["Title"] + " " + df["Text"]).str.strip()
df["Combined_Text"] = df["Combined_Text"].replace("", np.nan)
print("Non-empty Combined_Text rows:", df["Combined_Text"].notna().sum())

Non-empty Combined_Text rows: 16156


In [45]:
# --------------------------------------------
# 3. Helper functions
# --------------------------------------------
def normalize_cardiff_label(label):
    if label == "LABEL_0":
        return "NEGATIVE"
    if label == "LABEL_1":
        return "NEUTRAL"
    if label == "LABEL_2":
        return "POSITIVE"
    return label


def run_model_in_batches(texts, pipe, batch_size=32, max_len=256, desc=""):
    labels, scores = [], []
    clean_texts = ["" if pd.isna(t) else str(t).strip()[:max_len] for t in texts]
    total = len(clean_texts)
    for i in tqdm(range(0, total, batch_size), total=(total + batch_size - 1) // batch_size, desc=desc or "Batches"):
        batch = clean_texts[i:i + batch_size]
        if not any(batch):
            labels.extend([None] * len(batch))
            scores.extend([0.0] * len(batch))
            continue
        try:
            outputs = pipe(batch, truncation=True, batch_size=batch_size)
            for out in outputs:
                labels.append(out.get("label"))
                scores.append(round(float(out.get("score", 0.0)), 4))
        except Exception as exc:
            print(f"Batch {i} failed: {exc}")
            labels.extend([None] * len(batch))
            scores.extend([0.0] * len(batch))
    return labels, scores


def get_topic_keywords(topic_id, topic_model, top_n=5):
    if topic_id == -1:
        return "Outlier"
    topic_words = topic_model.get_topic(topic_id)
    if not topic_words:
        return None
    return ", ".join([f"{word}({round(score, 2)})" for word, score in topic_words[:top_n]])


def map_label_to_numeric(label):
    mapping = {
        "NEGATIVE": -1,
        "NEUTRAL": 0,
        "POSITIVE": 1,
        "Negative": -1,
        "Neutral": 0,
        "Positive": 1,
    }
    return mapping.get(label, np.nan)

In [46]:
# --------------------------------------------
# 4. Load models
# --------------------------------------------
print("Loading models...")
sentiment_pipe = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    device=device,
 )
emotion_pipe = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=device,
 )
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
topic_model = BERTopic(
    embedding_model=embedding_model,
    calculate_probabilities=True,
    umap_model=UMAP(n_neighbors=10, n_components=5, low_memory=True),
    verbose=True,
 )
print("Models ready.")

Loading models...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8413.05it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 22037.73it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4670.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-

Models ready.


In [47]:
# --------------------------------------------
# 5. Sentiment prediction (batched)
# --------------------------------------------
print("Running sentiment model...")
sent_labels, sent_scores = run_model_in_batches(
    df["Combined_Text"].tolist(),
    sentiment_pipe,
    batch_size=BATCH_SENTIMENT,
    max_len=MAX_TEXT_LEN,
    desc="Sentiment",
 )
df["Combined_Sentiment"] = [
    normalize_cardiff_label(x) if x is not None else None for x in sent_labels
 ]
df["Sentiment_Score"] = sent_scores
print(df[["Combined_Sentiment", "Sentiment_Score"]].head())

Running sentiment model...


Sentiment:   0%|          | 0/505 [00:00<?, ?it/s]

Sentiment: 100%|██████████| 505/505 [12:39<00:00,  1.50s/it]

  Combined_Sentiment  Sentiment_Score
0           POSITIVE           0.9814
1           POSITIVE           0.9893
2           POSITIVE           0.9923
3           POSITIVE           0.9743
4           POSITIVE           0.9883


In [48]:
# --------------------------------------------
# 6. Emotion prediction (batched)
# --------------------------------------------
print("Running emotion model...")
emotion_labels, _ = run_model_in_batches(
    df["Combined_Text"].tolist(),
    emotion_pipe,
    batch_size=BATCH_EMOTION,
    max_len=MAX_TEXT_LEN,
    desc="Emotion",
 )
df["Emotion"] = emotion_labels
print(df[["Emotion"]].head())

Running emotion model...


Emotion:   0%|          | 0/505 [00:00<?, ?it/s]

Emotion: 100%|██████████| 505/505 [06:09<00:00,  1.37it/s]

  Emotion
0     joy
1     joy
2     joy
3     joy
4     joy


In [49]:
# --------------------------------------------
# 7. Topic modeling with BERTopic
# --------------------------------------------
print("Running BERTopic (this may take a while)...")
topic_texts = df["Combined_Text"].fillna("No review text").tolist()
if TOPIC_SAMPLE_N is not None:
    topic_texts = topic_texts[:TOPIC_SAMPLE_N]
    df_subset = df.iloc[: len(topic_texts)].copy()
else:
    df_subset = df
topics, probs = topic_model.fit_transform(topic_texts)
df_subset["Dominant_Topic"] = topics
if probs is not None:
    topic_probs = []
    for p in probs:
        try:
            topic_probs.append(float(np.max(p)) if isinstance(p, np.ndarray) else float(p))
        except Exception:
            topic_probs.append(None)
    df_subset["Topic_Probability"] = topic_probs
else:
    df_subset["Topic_Probability"] = None
df_subset["Topic_Keywords"] = df_subset["Dominant_Topic"].apply(
    lambda x: get_topic_keywords(x, topic_model, top_n=5)
 )
topic_info = topic_model.get_topic_info()
print(topic_info.head(15))
if TOPIC_SAMPLE_N is not None:
    df.update(df_subset[["Dominant_Topic", "Topic_Probability", "Topic_Keywords"]])
else:
    df = df_subset

2026-03-24 21:18:43,207 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic (this may take a while)...


Batches: 100%|██████████| 505/505 [02:59<00:00,  2.82it/s]
2026-03-24 21:21:42,697 - BERTopic - Embedding - Completed ✓
2026-03-24 21:21:42,697 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-24 21:21:44,100 - BERTopic - Dimensionality - Completed ✓
2026-03-24 21:21:44,102 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-24 21:21:52,774 - BERTopic - Cluster - Completed ✓
2026-03-24 21:21:52,790 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-24 21:21:53,512 - BERTopic - Representation - Completed ✓


    Topic  Count                                     Name  \
0      -1   4029                         -1_the_to_and_it   
1       0   1604         0_elephants_park_safari_elephant   
2       1   1037                 1_beach_clean_waves_sand   
3       2    581               2_tea_factory_tour_process   
4       3    333                   3_tea_factory_tour_sri   
5       4    298                  4_lake_boat_walk_around   
6       5    246          5_buddha_statue_temple_buddhist   
7       6    238             6_gardens_garden_house_trees   
8       7    237                7_beach_sri_lanka_beaches   
9       8    231        8_forest_leeches_rainforest_guide   
10      9    207               9_island_coral_sharks_fish   
11     10    200       10_stupa_stupas_anuradhapura_brick   
12     11    177              11_tree_bodhi_oldest_sacred   
13     12    169          12_spices_spice_products_garden   
14     13    166  13_museum_building_exhibits_interesting   

                       

In [50]:
# --------------------------------------------
# 8. Manual topic-to-theme mapping
# --------------------------------------------
topic_theme_map = {
    0: "Scenery",
    1: "Cultural Experience",
    2: "Service Quality",
    3: "Crowding & Pricing",
    4: "Food & Hospitality",
    5: "Wildlife & Nature"
}
df["Review_Theme"] = df["Dominant_Topic"].map(topic_theme_map).fillna("Other")
print(df[["Dominant_Topic", "Review_Theme"]].head())

   Dominant_Topic         Review_Theme
0              -1                Other
1              83                Other
2              83                Other
3               1  Cultural Experience
4               1  Cultural Experience


In [51]:
# --------------------------------------------
# 9. Sentiment-rating gap
# --------------------------------------------
df["rating_class_num"] = df["Rating_Class"].apply(map_label_to_numeric)
df["sentiment_num"] = df["Combined_Sentiment"].apply(map_label_to_numeric)
df["Sentiment_Rating_Gap"] = (df["rating_class_num"] - df["sentiment_num"]).abs()
df.drop(columns=["rating_class_num", "sentiment_num"], inplace=True)
print(df[["Rating_Class", "Combined_Sentiment", "Sentiment_Rating_Gap"]].head())

  Rating_Class Combined_Sentiment  Sentiment_Rating_Gap
0     Positive           POSITIVE                     0
1     Positive           POSITIVE                     0
2     Positive           POSITIVE                     0
3     Positive           POSITIVE                     0
4     Positive           POSITIVE                     0


In [52]:
# --------------------------------------------
# 9b. Ensure all column segments start capitalized
# --------------------------------------------
def _capitalize_segments(name: str) -> str:
    return "_".join([part[:1].upper() + part[1:] if part else part for part in name.split("_")])

df.rename(columns={c: _capitalize_segments(c) for c in df.columns}, inplace=True)

In [53]:
# --------------------------------------------
# 10. Save enriched dataset (overwrite input)
# --------------------------------------------
df = df.drop(columns=["Unnamed: 0", "Text", "Title"], errors="ignore")
df.to_csv(INPUT_FILE, index=False)
print(f"Saved: {INPUT_FILE}")
print("Final shape:", df.shape)
print("Columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

Saved: Processed_Reviews.csv
Final shape: (16156, 28)
Columns:
 1. Location_Name
 2. Located_City
 3. Province
 4. District
 5. Location_Type
 6. User_Locale
 7. User_Country
 8. User_Region
 9. Travel_Date_Month
10. Travel_Date_Year
11. Published_Date_Month
12. Published_Date_Year
13. User_Contributions
14. Rating
15. Helpful_Votes
16. Review_Length
17. Title_Length
18. Rating_Class
19. Review_Delay_Days
20. Combined_Text
21. Combined_Sentiment
22. Sentiment_Score
23. Emotion
24. Dominant_Topic
25. Topic_Probability
26. Topic_Keywords
27. Review_Theme
28. Sentiment_Rating_Gap
